In [1]:
import pandas as pd
import numpy as np
import re, os
from pathlib import Path

DATA_PATH = Path(r"C:\Users\oozgen\Desktop\phd\dataset\training\SOD1_cleaned_v3.csv")  # change if needed

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

df.head()

,ISIS,Sequence,Modification,Location,Chemical_Pattern,Linkage,Linkage_Location,Smiles,Inhibition(%),seq_length
0,146143,TCAGCACGCACACGGCCTTC,MOE/5-methylcytosines/deoxy,0?1?2?3?4?15?16?17?18?19/C/else,MMMMMddddddddddMMMMM,phosphorothioate,else,COCCO[C@@H]1[C@H](O)[C@@H](C[O]P(=O)([S-])O[C@...,88,20
1,150438,GGTTCCGAGGACTGCAACGG,MOE/5-methylcytosines/deoxy,0?1?2?3?4?15?16?17?18?19/C/else,MMMMMddddddddddMMMMM,phosphorothioate,else,COCCO[C@@H]1[C@H](OP(=O)([S-])[O]C[C@H]2O[C@@H...,35,20
2,150439,TCCTGGTTCCGAGGACTGCA,MOE/5-methylcytosines/deoxy,0?1?2?3?4?15?16?17?18?19/C/else,MMMMMddddddddddMMMMM,phosphorothioate,else,COCCO[C@@H]1[C@H](OP(=O)([S-])[O]C[C@H]2O[C@@H...,24,20
3,150440,GAGGTCCTGGTTCCGAGGAC,MOE/5-methylcytosines/deoxy,0?1?2?3?4?15?16?17?18?19/C/else,MMMMMddddddddddMMMMM,phosphorothioate,else,COCCO[C@@H]1[C@H](O)[C@@H](C[O]P(=O)([S-])O[C@...,31,20
4,150442,GTCGCCATAACTCGCTAGGC,MOE/5-methylcytosines/deoxy,0?1?2?3?4?15?16?17?18?19/C/else,MMMMMddddddddddMMMMM,phosphorothioate,else,COCCO[C@@H]1[C@H](O)[C@@H](C[O]P(=O)([S-])O[C@...,90,20


In [2]:
# -----------------------------
# 1. Recalculate length fields
# -----------------------------

df["Sequence"] = df["Sequence"].astype(str).str.upper().str.strip()
df["Chemical_Pattern"] = df["Chemical_Pattern"].astype(str).str.strip()

df["seq_length_recalc"] = df["Sequence"].str.len()
df["chem_pattern_length"] = df["Chemical_Pattern"].str.len()

# Compare original seq_length with recalculated value, if present
if "seq_length" in df.columns:
    df["seq_length_original"] = df["seq_length"]
    df["seq_length"] = df["seq_length_recalc"]
else:
    df["seq_length"] = df["seq_length_recalc"]

# Basic sequence validity
valid_bases = set("ACGT")
df["valid_sequence"] = df["Sequence"].apply(lambda x: set(x).issubset(valid_bases))

# Pattern validity based on your current notation:
# M = MOE
# C = cEt
# d = DNA/deoxy
valid_pattern_symbols = set("MCd")
df["valid_chemical_pattern"] = df["Chemical_Pattern"].apply(
    lambda x: set(x).issubset(valid_pattern_symbols)
)

# Length consistency
df["lengths_match"] = (
    (df["seq_length_recalc"] == df["chem_pattern_length"]) &
    df["valid_sequence"] &
    df["valid_chemical_pattern"]
)

length_issues = df.loc[
    ~df["lengths_match"],
    [
        "ISIS",
        "Sequence",
        "Chemical_Pattern",
        "seq_length_original" if "seq_length_original" in df.columns else "seq_length",
        "seq_length_recalc",
        "chem_pattern_length",
        "valid_sequence",
        "valid_chemical_pattern",
        "lengths_match",
    ],
]

print(f"Total rows: {len(df)}")
print(f"Rows with length / sequence / pattern issues: {len(length_issues)}")

length_issues

Total rows: 1026
Rows with length / sequence / pattern issues: 0


,ISIS,Sequence,Chemical_Pattern,seq_length_original,seq_length_recalc,chem_pattern_length,valid_sequence,valid_chemical_pattern,lengths_match


In [ ]:
if len(length_issues) > 0:
    length_issues.to_csv("length_validation_issues.csv", index=False)
    print("Saved: length_validation_issues.csv")